# `ptof_obs_behavioral_correlation`

## What this notebook does
Detects handover delivery failures from the ISH system-of-record:
1. **Handover delivery failures** (CRITICAL) — a shift handover email silently failed to send
2. **Handover delivery rate** (CRITICAL) — 7-day rolling failure rate exceeds threshold

These are behavioral signals that live outside the LLM output itself — a handover email that
never reached the distribution list is arguably a worse defect than any content-quality issue.

## Position in the pipeline
- **Job:** `obs_fresh_scan` task `05_behavioral_correlation`, runs after `01_bronze_projections`,
  before `06_alert`.
- **Upstream:** reads `v_ish_bronze` (built by `ptof_obs_bronze_projection`, pointing at prod
  `mq_gmdf_dp_prd.oil.ptof_ish_audit` — schema identical to dev, confirmed).
- **Downstream:** `ptof_obs_alert.ipynb` reads `handover_delivery_failures` (detector
  `handover_delivery`, CRITICAL) and `handover_delivery_rate` (detector `handover_delivery_rate`,
  CRITICAL) directly, with its own inline threshold filter — no separate findings table.

## Tables/views touched
- **Reads:** `v_ish_bronze`
- **Writes:** `handover_delivery_failures`, `handover_delivery_rate`

## Dropped detectors (prod migration 2026-09-10)
- `ish_entity_dim` — only consumer (rapid_human_correction) was dropped
- `rapid_human_correction` — 6 rows total historically, dormant, research-quality signal

## Dropped tables (2026-09-21, dead-code audit)
- `handover_delivery_rate_findings` — duplicate of the threshold check `ptof_obs_alert.ipynb`
  already applies inline against `handover_delivery_rate`; nothing ever read this table.

## Investigated and closed: "blank handover email body" gap (2026-09-21)
Question raised: `handover_delivery_failures` only checks the `sent` boolean (did the SMTP send
succeed or fail) — it never looks at whether a *successfully sent* handover email's actual body
text was blank/empty. Could a handover go out with a blank body while `sent=true`, slipping past
every detector?

**Checked against live data and closed as not a real gap, no new detector added:**
- The handover email's body (`after_v:content` in `v_ish_bronze`, entity_type='HandoverEmail') is
  structurally the *same* generated content as the `summary` capability's LLM output in
  `v_llm_bronze` — e.g. `content.summary` in the email is the same text as `how_we_ran` in the
  LLM output row. It is not populated independently; there's no separate code path that could
  produce a blank email body while the LLM output itself is non-blank.
- Live query across all 49 `sent=true` HandoverEmail rows (2026-08-27 through 2026-09-21):
  0 rows had a blank/empty content object, 0 had a blank subject. Exactly 1 row had a blank
  `content.summary` field (2026-08-27 22:34 UTC) — but that predates `v_llm_bronze`'s earliest
  `summary`-capability row (2026-08-28 10:21 UTC) by ~12 hours, i.e. it's an artifact of the LLM
  output audit table not having started logging yet, not a live divergence between the two.
- Conclusion: any blank-content failure on this path is already caught upstream by
  `blank_output_findings` in `ptof_obs_mal_output.ipynb` (which special-cases the `summary`
  capability's `how_we_ran` field as CRITICAL, unconditional floor) before the email is ever
  composed. Adding a separate blank-body check here would duplicate that detector against a
  failure mode with zero empirical occurrences and no structural path to occur independently.
  Revisit only if the handover email's body generation is ever decoupled from the `summary`
  capability's LLM output (e.g. a template/caching layer inserted between them).

In [0]:
%sql
-- handover_delivery_failures — a handover that silently failed to send.
-- Two sent=false reasons exist in this data:
--   'email disabled in this environment (EMAIL_ENABLED=false)' -> config gate, excluded
--   'SMTP send failed: [Errno 11001] getaddrinfo failed'       -> real delivery failure
-- In a shift-handover system a handover that never reached the distribution list is arguably a
-- worse defect than a mildly ungrounded number, and nothing detected it before.
-- recipients/subject are masked (local-part before @ replaced with ***, subject bounded to
-- 200 chars) -- both may contain real distribution-list addresses / free text, never stored raw.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.handover_delivery_failures AS
SELECT
    i.id                              AS ish_row_id,
    i.ts                              AS attempted_at,
    i.shift_date_norm                 AS shift_date,
    i.shift_type_norm                 AS shift_type,
    i.batch_id_norm                   AS batch_nbr,
    i.after_v:reason::string           AS failure_reason,
    regexp_replace(cast(i.after_v:to AS STRING), '([^,;\\s@]+)@', '***@') AS recipients,
    i.after_v:trigger::string          AS trigger_type,
    substring(regexp_replace(i.after_v:subject::string, '([^,;\\s@]+)@', '***@'), 1, 200) AS subject,
    current_timestamp()                AS detected_at
FROM mq_gmdf_dev.oil_obs.v_ish_bronze i
WHERE i.entity_type = 'HandoverEmail'
  AND i.after_v:sent::boolean = false
  AND i.is_email_disabled_gate = false;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- handover_delivery_rate — 7-day rolling, single row.
-- Daily rates are unusable: ~2 attempts/day means one failure swings the rate to 33% or 100%.
-- Baseline established 2026-08-20: 15 of 162 attempts failed since Jun 19 = 9.3%.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.handover_delivery_rate AS
SELECT
    count_if(after_v:sent::boolean = true)  AS sent_ok,
    count_if(after_v:sent::boolean = false AND is_email_disabled_gate = false) AS failed,
    count_if(is_email_disabled_gate)        AS gated,
    round(count_if(after_v:sent::boolean = false AND is_email_disabled_gate = false) * 100.0
          / nullif(count_if(after_v:sent::boolean = true)
                   + count_if(after_v:sent::boolean = false AND is_email_disabled_gate = false), 0), 1)
      AS failure_pct_7d,
    max(ts) AS last_attempt
FROM mq_gmdf_dev.oil_obs.v_ish_bronze
WHERE entity_type = 'HandoverEmail'
  AND ts >= current_timestamp() - INTERVAL 7 DAYS;

num_affected_rows,num_inserted_rows
